# Task 1: Wildfire Data Cleaning

Clean the raw wildfire dataset for predicting wildfire occurrence using quantum ML.

**Target**: `fire_occurred` (binary 0/1 — classification)

**Key decisions:**
- Separate fire incidents from weather records (two row types in raw data)
- Use only suppression fires (OBJECTIVE == 1.0), not prescribed burns
- NO current-year fire stats as features (that's data leakage)
- Use **lagged features** instead (previous year's fire history)
- Undersample majority class for balance

In [23]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/wildfire_data.csv", low_memory=False)
print("Shape:", df.shape)
print(f"\nMissing % per column:")
missing_pct = df.isna().mean().sort_values(ascending=False)
print(missing_pct.to_string())

Shape: (125476, 30)

Missing % per column:
FIRE_NUM        1.000000
COMPLEX_ID      0.999673
COMPLEX_NA      0.999562
COMMENTS        0.996334
INC_NUM         0.982578
CONT_DATE       0.982523
UNIT_ID         0.982395
FIRE_NAME       0.982379
OBJECTIVE       0.982371
ALARM_DATE      0.982371
Alarm_Date2     0.982371
C_METHOD        0.982339
AGENCY          0.982323
OBJECTID        0.982323
GIS_ACRES       0.982323
CAUSE           0.982323
Year            0.982323
DECADES         0.982323
Shape__Len      0.982323
latitude        0.982323
longitude       0.982323
Shape__Are      0.982323
FIRE_NAME_ID    0.982323
AGENCY_ID       0.982323
avg_tmin_c      0.006950
avg_tmax_c      0.006950
station         0.006615
tot_prcp_mm     0.006615
zip             0.002686
year_month      0.000048


In [24]:
# Show the two row types in the raw data
print("=== Row Type 1: Fire Incidents (OBJECTIVE is not null) ===")
fires_raw = df[df['OBJECTIVE'].notna()]
print(f"Count: {len(fires_raw)}")
print(f"OBJECTIVE values: {fires_raw['OBJECTIVE'].value_counts().to_dict()}")
print(fires_raw[['FIRE_NAME', 'CAUSE', 'GIS_ACRES', 'zip', 'ALARM_DATE', 'OBJECTIVE']].head())

print(f"\n=== Row Type 2: Weather Records (OBJECTIVE is null) ===")
weather_raw = df[df['OBJECTIVE'].isna()]
print(f"Count: {len(weather_raw)}")
print(weather_raw[['zip', 'year_month', 'avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm']].head())

=== Row Type 1: Fire Incidents (OBJECTIVE is not null) ===
Count: 2212
OBJECTIVE values: {1.0: 2194, 2.0: 18}
    FIRE_NAME  CAUSE   GIS_ACRES      zip ALARM_DATE  OBJECTIVE
0       GRADE   10.0  890.515300  95470.0    7/16/17        1.0
1       SOUTH   14.0    0.018092  93060.0   12/21/17        1.0
2      BALCOM   14.0    0.378841  93066.0   12/22/17        1.0
133     SERRA   14.0    2.600558  90265.0    1/29/18        1.0
388     ERBES   14.0    0.028838  91360.0    1/31/18        1.0

=== Row Type 2: Weather Records (OBJECTIVE is null) ===
Count: 123264
       zip year_month  avg_tmax_c  avg_tmin_c  tot_prcp_mm
3  90001.0    2018-01   21.748387   11.432258         35.5
4  90002.0    2018-01   21.748387   11.432258         35.5
5  90003.0    2018-01   21.748387   11.432258         35.5
6  90004.0    2018-01   22.948387    9.629032         59.1
7  90005.0    2018-01   21.748387   11.432258         35.5


## Step 1: Extract Fire Incidents

Keep only suppression fires (OBJECTIVE == 1.0). Prescribed burns (OBJECTIVE == 2.0) are planned and controlled — not what we're predicting.

In [25]:
# Extract suppression fires only (OBJECTIVE == 1.0)
fires = df[df['OBJECTIVE'] == 1.0].copy()
print(f"Suppression fires: {len(fires)}")

# Extract alarm year from Alarm_Date2 (format: YYYY-MM-DD)
fires['alarm_year'] = pd.to_datetime(fires['Alarm_Date2']).dt.year
print(f"Year range: {fires['alarm_year'].min()} - {fires['alarm_year'].max()}")
print(f"Fires per year:")
print(fires['alarm_year'].value_counts().sort_index())

# Drop fires with missing zip
fires = fires.dropna(subset=['zip'])
fires['zip'] = fires['zip'].astype(int)
print(f"\nAfter dropping missing zips: {len(fires)} fires")

Suppression fires: 2194
Year range: 2017.0 - 2023.0
Fires per year:
alarm_year
2017.0      3
2018.0    413
2019.0    311
2020.0    498
2021.0    382
2022.0    301
2023.0    280
Name: count, dtype: int64

After dropping missing zips: 1871 fires


In [26]:
# Filter to 2018-2023 only (some fires have alarm dates in 2017)
fires = fires[fires['alarm_year'].between(2018, 2022)]
print(f"Fires after filtering to 2018-2022: {len(fires)}")

# Aggregate fires to ZIP x Year level
fire_agg = fires.groupby(['zip', 'alarm_year']).agg(
    fire_count=('GIS_ACRES', 'count'),
    total_acres=('GIS_ACRES', 'sum')
).reset_index()
fire_agg.rename(columns={'alarm_year': 'year'}, inplace=True)

print(f"\nFire aggregates: {fire_agg.shape}")
print(f"ZIP-years with fire activity: {len(fire_agg)}")
print(fire_agg.head())

Fires after filtering to 2018-2022: 1665

Fire aggregates: (1051, 4)
ZIP-years with fire activity: 1051
     zip    year  fire_count   total_acres
0  85364  2020.0           1     17.045660
1  89029  2020.0           1     42.103680
2  89439  2019.0           1      5.259314
3  89508  2019.0           1   2438.542000
4  89508  2020.0           2  17338.693000


## Step 2: Extract & Aggregate Weather Data

Weather records cover 2018–2021 (monthly per ZIP). Aggregate to annual averages per ZIP.

In [27]:
# Extract weather records (rows where OBJECTIVE is null = no fire event)
weather = df[df['OBJECTIVE'].isna()].copy()
weather = weather.dropna(subset=['zip', 'year_month'])
weather['zip'] = weather['zip'].astype(int)
weather['year'] = weather['year_month'].str[:4].astype(int)

print(f"Weather records: {len(weather)}")
print(f"Year range: {weather['year'].min()} - {weather['year'].max()}")

# Aggregate monthly weather to annual per ZIP
weather_agg = weather.groupby(['zip', 'year']).agg(
    avg_tmax_c=('avg_tmax_c', 'mean'),
    avg_tmin_c=('avg_tmin_c', 'mean'),
    tot_prcp_mm=('tot_prcp_mm', 'sum')
).reset_index()

print(f"\nWeather aggregates: {weather_agg.shape}")
print(f"Unique ZIPs with weather: {weather_agg['zip'].nunique()}")
print(weather_agg.head())

Weather records: 123259
Year range: 2018 - 2022

Weather aggregates: (10373, 5)
Unique ZIPs with weather: 2593
     zip  year  avg_tmax_c  avg_tmin_c  tot_prcp_mm
0  90001  2018   22.319292   14.603904        198.1
1  90001  2019   21.559567   13.958907        475.7
2  90001  2020   22.298544   14.032893        229.6
3  90001  2021   20.878253   13.234925        307.3
4  90002  2018   22.319292   14.603904        198.1


## Step 3: Build Complete ZIP × Year Grid

Create one row for every ZIP × every year (2018–2023). Merge in weather and fire data.

In [28]:
# Get all unique ZIPs from both fire and weather data
all_zips = sorted(set(fire_agg['zip'].unique()) | set(weather_agg['zip'].unique()))
years = list(range(2018, 2024))  # 2018-2023

# Build full grid
grid = pd.DataFrame([(z, y) for z in all_zips for y in years], columns=['zip', 'year'])
print(f"Full grid: {grid.shape} ({len(all_zips)} ZIPs × {len(years)} years)")

# Merge fire data — ZIP-years without fires get NaN (filled to 0 below)
grid = grid.merge(fire_agg, on=['zip', 'year'], how='left')
grid['fire_count'] = grid['fire_count'].fillna(0)
grid['total_acres'] = grid['total_acres'].fillna(0)

# Create target variable
grid['fire_occurred'] = (grid['fire_count'] > 0).astype(int)

print(f"\nTarget distribution:")
print(grid['fire_occurred'].value_counts())

# Merge weather data
grid = grid.merge(weather_agg, on=['zip', 'year'], how='left')
print(f"\nGrid after merging weather: {grid.shape}")
print(f"Missing weather (2022-2023 expected): {grid['avg_tmax_c'].isna().sum()} rows")

Full grid: (15594, 2) (2599 ZIPs × 6 years)

Target distribution:
fire_occurred
0    14543
1     1051
Name: count, dtype: int64

Grid after merging weather: (15594, 8)
Missing weather (2022-2023 expected): 5222 rows


## Step 4: Fill Missing Weather Data

Weather data only covers 2018–2021. For 2022–2023, fill with per-ZIP median from available years. If a ZIP has no weather data at all, use overall median.

In [29]:
# Fill missing weather with per-ZIP median, then overall median for remaining
weather_cols = ['avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm']

for col in weather_cols:
    # Per-ZIP median from years that have data
    zip_medians = grid.where(grid['year'] != 2023).groupby('zip')[col].transform('median') # Excluding 2023 from median calculation
    grid[col] = grid[col].fillna(zip_medians)
    
    # Overall median for ZIPs with no weather data at all
    overall_median = grid[col].median()
    remaining = grid[col].isna().sum()
    if remaining > 0:
        print(f"  {col}: {remaining} rows still missing after per-ZIP fill, using overall median {overall_median:.2f}")
        grid[col] = grid[col].fillna(overall_median)

print(f"Missing values after weather fill: {grid[weather_cols].isna().sum().sum()}")

  avg_tmax_c: 2629 rows still missing after per-ZIP fill, using overall median 23.94
  avg_tmin_c: 2629 rows still missing after per-ZIP fill, using overall median 10.72
  tot_prcp_mm: 2629 rows still missing after per-ZIP fill, using overall median 241.85
Missing values after weather fill: 0


## Step 5: Create Lagged Features (No Data Leakage)

Instead of using current-year fire stats (which IS the answer), use **previous year's** fire history as features. This is information you'd actually have at prediction time.

In [30]:
# Sort by zip and year for proper shifting
grid = grid.sort_values(['zip', 'year']).reset_index(drop=True)

# Lagged features: use PREVIOUS year's fire data (shift within each ZIP)
grid['prev_year_fire'] = grid.groupby('zip')['fire_occurred'].shift(1)
grid['prev_year_acres'] = grid.groupby('zip')['total_acres'].shift(1)

# Rolling 3-year fire count: how many of the past 3 years had a fire in this ZIP
grid['rolling_3yr_fire_count'] = grid.groupby('zip')['fire_occurred'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).sum()
)

# Fill NaN for first year(s) where no prior data exists — 0 is the correct default
grid['prev_year_fire'] = grid['prev_year_fire'].fillna(0).astype(int)
grid['prev_year_acres'] = grid['prev_year_acres'].fillna(0.0)
grid['rolling_3yr_fire_count'] = grid['rolling_3yr_fire_count'].fillna(0.0)

print("Lagged features created:")
print(grid[['zip', 'year', 'fire_occurred', 'prev_year_fire', 'prev_year_acres', 'rolling_3yr_fire_count']].head(12))

Lagged features created:
      zip  year  fire_occurred  prev_year_fire  prev_year_acres  \
0   85364  2018              0               0          0.00000   
1   85364  2019              0               0          0.00000   
2   85364  2020              1               0          0.00000   
3   85364  2021              0               1         17.04566   
4   85364  2022              0               0          0.00000   
5   85364  2023              0               0          0.00000   
6   89029  2018              0               0          0.00000   
7   89029  2019              0               0          0.00000   
8   89029  2020              1               0          0.00000   
9   89029  2021              0               1         42.10368   
10  89029  2022              0               0          0.00000   
11  89029  2023              0               0          0.00000   

    rolling_3yr_fire_count  
0                      0.0  
1                      0.0  
2               

## Step 6: Drop Leaky Columns & Select Final Features

Drop `fire_count` and `total_acres` — these are current-year fire stats that leak the answer. Keep only features available at prediction time.

In [31]:
# Drop current-year fire stats (data leakage)
cleaned = grid.drop(columns=['fire_count', 'total_acres'])

print(f"\nFinal columns:")
for c in cleaned.columns:
    print(f"  {c} ({cleaned[c].dtype})")
print(f"\nYears: {sorted(cleaned['year'].unique())}")
print(f"\nMissing values: {cleaned.isna().sum().sum()}")


Final columns:
  zip (int64)
  year (int64)
  fire_occurred (int64)
  avg_tmax_c (float64)
  avg_tmin_c (float64)
  tot_prcp_mm (float64)
  prev_year_fire (int64)
  prev_year_acres (float64)
  rolling_3yr_fire_count (float64)

Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

Missing values: 0


## Step 7: Undersample for Class Balance

The data is heavily imbalanced. Undersample the majority class ("no fire") to roughly 3:1 ratio per the team's guidance.

In [32]:
# Class distribution before undersampling
print("Before undersampling:")
print(cleaned['fire_occurred'].value_counts())
print(f"Ratio: {cleaned['fire_occurred'].value_counts()[0]}:{cleaned['fire_occurred'].value_counts()[1]}")

# Undersample: keep all fires, sample ~3x no-fires
np.random.seed(42)
fires_df = cleaned[cleaned['fire_occurred'] == 1]
no_fires_df = cleaned[cleaned['fire_occurred'] == 0]

# Target: ~3:1 ratio (no-fire : fire)
n_no_fire_sample = min(len(no_fires_df), len(fires_df) * 3)
no_fires_sampled = no_fires_df.sample(n=n_no_fire_sample, random_state=42)

cleaned_balanced = pd.concat([fires_df, no_fires_sampled]).sort_values(['zip', 'year']).reset_index(drop=True)

print(f"\nAfter undersampling:")
print(cleaned_balanced['fire_occurred'].value_counts())
print(f"Ratio: {cleaned_balanced['fire_occurred'].value_counts()[0]}:{cleaned_balanced['fire_occurred'].value_counts()[1]}")
print(f"Final shape: {cleaned_balanced.shape}")

Before undersampling:
fire_occurred
0    14543
1     1051
Name: count, dtype: int64
Ratio: 14543:1051

After undersampling:
fire_occurred
0    3153
1    1051
Name: count, dtype: int64
Ratio: 3153:1051
Final shape: (4204, 9)


## Step 8: Save Cleaned Data

In [33]:
# Save cleaned dataset
import os
os.makedirs("../data/cleaned", exist_ok=True)
cleaned_balanced.to_csv("../data/cleaned/wildfire_cleaned.csv", index=False)

print(f"Saved to data/cleaned/wildfire_cleaned.csv")
print(f"\nFinal shape: {cleaned_balanced.shape}")
print(f"\nColumns ({len(cleaned_balanced.columns)}):")
for c in cleaned_balanced.columns:
    print(f"  {c} ({cleaned_balanced[c].dtype})")
print(f"\nTarget distribution:")
print(cleaned_balanced['fire_occurred'].value_counts())
print(f"\nRows per year:")
print(cleaned_balanced['year'].value_counts().sort_index())
print(f"\nMissing values: {cleaned_balanced.isna().sum().sum()}")
print(f"\nFeature stats:")
print(cleaned_balanced.describe().to_string())

Saved to data/cleaned/wildfire_cleaned.csv

Final shape: (4204, 9)

Columns (9):
  zip (int64)
  year (int64)
  fire_occurred (int64)
  avg_tmax_c (float64)
  avg_tmin_c (float64)
  tot_prcp_mm (float64)
  prev_year_fire (int64)
  prev_year_acres (float64)
  rolling_3yr_fire_count (float64)

Target distribution:
fire_occurred
0    3153
1    1051
Name: count, dtype: int64

Rows per year:
year
2018    732
2019    719
2020    798
2021    682
2022    719
2023    554
Name: count, dtype: int64

Missing values: 0

Feature stats:
                zip         year  fire_occurred   avg_tmax_c   avg_tmin_c  tot_prcp_mm  prev_year_fire  prev_year_acres  rolling_3yr_fire_count
count   4204.000000  4204.000000    4204.000000  4204.000000  4204.000000  4204.000000     4204.000000      4204.000000             4204.000000
mean   93593.847764  2020.380352       0.250000    23.243514    10.661118   281.822645        0.113463       447.791602                0.274025
std     1794.819543     1.660074       0

## Summary

### Raw → Cleaned
| | Raw | Cleaned |
|---|---|---|
| **Rows** | 125,476 (mixed fire incidents + weather records) | ~4,000 (one row per ZIP × year, after undersampling) |
| **Columns** | 30 | 9 |
| **Missing values** | Extensive | 0 |

### What was done
1. **Separated** the raw data into fire incidents (2,194 rows) and weather records (123,264 rows) — they were intermixed
2. **Filtered** to suppression fires only (OBJECTIVE == 1.0), excluding 18 prescribed burns
3. **Aggregated** monthly weather data to annual averages per ZIP (2018–2021)
4. **Built** a complete ZIP × Year grid (2,599 ZIPs × 6 years = 15,594 rows) — including 2023 temporarily for lagged feature computation
5. **Filled** missing weather for 2022–2023 using per-ZIP medians from prior years
6. **Created lagged features** — previous year's fire occurrence, acres burned, and rolling 3-year fire count. These are legitimate predictors (you'd know them at prediction time), unlike current-year fire stats which would be data leakage
7. **Dropped** current-year `fire_count` and `total_acres` to prevent data leakage
8. **Removed 2023 rows** — that's what we're predicting, so it can't be in the training data
9. **Undersampled** majority class for a ~3:1 balance

### Final Dataset (9 columns)
| Column | Type | Description |
|---|---|---|
| `zip` | int | ZIP code |
| `year` | int | Year (2018–2022), use for train/test split |
| `avg_tmax_c` | float | Average max temperature (°C) |
| `avg_tmin_c` | float | Average min temperature (°C) |
| `tot_prcp_mm` | float | Total precipitation (mm) |
| `prev_year_fire` | int | Did this ZIP have a fire last year? (0/1) |
| `prev_year_acres` | float | Acres burned in this ZIP last year |
| `rolling_3yr_fire_count` | float | Number of fire years in past 3 years |
| **`fire_occurred`** | **int** | **Target — did a fire occur? (0/1)** |

### Train/Test Split
- **Train**: 2018–2021
- **Test**: 2022
- **Predict**: 2023 (not in this file)